# Session 1: General Data Protection Regulation (GDPR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/05-cyber-security-frameworks/code/session-01-gdpr.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-01_principles_audit.py`

In [1]:
"""Session 1 - Audit a record of processing activities against the GDPR principles.

A "record of processing activities" (ROPA) is the list every organisation keeps of
WHAT personal data it processes, WHY, on WHICH lawful basis, for HOW LONG and HOW it
is protected. This script checks each activity against the seven principles of
Article 5 and the six lawful bases of Article 6, then prints a findings report.

All organisations, people and data below are fictional teaching examples.
Run:  python session-01_principles_audit.py
"""

LAWFUL_BASES = {"consent", "contract", "legal_obligation",
                "vital_interests", "public_task", "legitimate_interests"}

# Each activity is one row of the ROPA of a fictional online bookshop.
ROPA = [
    {"name": "Order fulfilment", "purpose": "deliver books to customers",
     "basis": "contract", "fields": ["name", "address", "phone", "email"],
     "needed": ["name", "address", "phone", "email"],
     "retention_days": 2190, "retention_reason": "tax law requires 6 years",
     "encrypted": True, "access_roles": ["warehouse", "support"],
     "privacy_notice": True, "last_accuracy_check_days": 30, "dpia_done": True},
    {"name": "Marketing newsletter", "purpose": "send offers by email",
     "basis": "consent", "consent_recorded": False, "fields": ["email", "name", "date_of_birth"],
     "needed": ["email", "name"], "retention_days": None, "retention_reason": "",
     "encrypted": True, "access_roles": ["marketing"],
     "privacy_notice": True, "last_accuracy_check_days": 400, "dpia_done": False},
    {"name": "CCTV in warehouse", "purpose": "prevent theft",
     "basis": "legitimate_interests", "lia_done": True, "fields": ["video_image"],
     "needed": ["video_image"], "retention_days": 30, "retention_reason": "investigate incidents",
     "encrypted": False, "access_roles": ["security", "everyone"],
     "privacy_notice": False, "last_accuracy_check_days": 0, "dpia_done": True},
    {"name": "Customer profiling", "purpose": "",
     "basis": "unknown", "fields": ["browsing_history", "location", "email", "religion"],
     "needed": ["browsing_history"], "retention_days": 3650, "retention_reason": "might be useful",
     "encrypted": False, "access_roles": ["analytics"],
     "privacy_notice": False, "last_accuracy_check_days": 90, "dpia_done": False},
]

SPECIAL_CATEGORY = {"religion", "health", "ethnicity", "biometric", "sexual_orientation",
                    "political_opinion", "trade_union", "genetic"}


def audit(act):
    """Return a list of (principle, finding) tuples for one activity."""
    f = []
    # 1. Lawfulness, fairness and transparency
    if act["basis"] not in LAWFUL_BASES:
        f.append(("Lawfulness", "no valid Article 6 lawful basis recorded"))
    if act["basis"] == "consent" and not act.get("consent_recorded"):
        f.append(("Lawfulness", "consent is the basis but no proof of consent is stored"))
    if act["basis"] == "legitimate_interests" and not act.get("lia_done"):
        f.append(("Lawfulness", "legitimate interests used without a balancing test (LIA)"))
    if not act["privacy_notice"]:
        f.append(("Transparency", "people are not told about this processing"))
    # 2. Purpose limitation
    if not act["purpose"].strip():
        f.append(("Purpose limitation", "purpose is blank or vague"))
    # 3. Data minimisation
    extra = sorted(set(act["fields"]) - set(act["needed"]))
    if extra:
        f.append(("Data minimisation", "collects fields not needed: " + ", ".join(extra)))
    special = sorted(set(act["fields"]) & SPECIAL_CATEGORY)
    if special:
        f.append(("Lawfulness", "special-category data (" + ", ".join(special) + ") needs an Article 9 condition"))
    # 4. Accuracy
    if act["last_accuracy_check_days"] > 365:
        f.append(("Accuracy", f"data not reviewed for {act['last_accuracy_check_days']} days"))
    # 5. Storage limitation
    if act["retention_days"] is None:
        f.append(("Storage limitation", "no retention period defined"))
    elif "might" in act["retention_reason"] or not act["retention_reason"]:
        f.append(("Storage limitation", f"{act['retention_days']} days kept without a real reason"))
    # 6. Integrity and confidentiality (security)
    if not act["encrypted"]:
        f.append(("Integrity & confidentiality", "data is not encrypted at rest"))
    if "everyone" in act["access_roles"]:
        f.append(("Integrity & confidentiality", "access not restricted (role 'everyone')"))
    # 7. Accountability
    if not act["dpia_done"] and (special or "profiling" in act["name"].lower()):
        f.append(("Accountability", "high-risk processing without a DPIA"))
    return f


def main():
    total = 0
    print("GDPR principles audit - fictional online bookshop")
    print("=" * 62)
    for act in ROPA:
        findings = audit(act)
        total += len(findings)
        status = "PASS" if not findings else f"{len(findings)} issue(s)"
        print(f"\n[{status}] {act['name']}  (basis: {act['basis']})")
        for principle, text in findings:
            print(f"   - {principle:<28} {text}")
    print("\n" + "=" * 62)
    by_principle = {}
    for act in ROPA:
        for p, _ in audit(act):
            by_principle[p] = by_principle.get(p, 0) + 1
    print(f"Total findings: {total}")
    for p, n in sorted(by_principle.items(), key=lambda x: (-x[1], x[0])):
        print(f"   {p:<28} {'#' * n} {n}")


if __name__ == "__main__":
    main()

GDPR principles audit - fictional online bookshop

[PASS] Order fulfilment  (basis: contract)

[4 issue(s)] Marketing newsletter  (basis: consent)
   - Lawfulness                   consent is the basis but no proof of consent is stored
   - Data minimisation            collects fields not needed: date_of_birth
   - Accuracy                     data not reviewed for 400 days
   - Storage limitation           no retention period defined

[3 issue(s)] CCTV in warehouse  (basis: legitimate_interests)
   - Transparency                 people are not told about this processing
   - Integrity & confidentiality  data is not encrypted at rest
   - Integrity & confidentiality  access not restricted (role 'everyone')

[8 issue(s)] Customer profiling  (basis: unknown)
   - Lawfulness                   no valid Article 6 lawful basis recorded
   - Transparency                 people are not told about this processing
   - Purpose limitation           purpose is blank or vague
   - Data minimisation

## `session-01_dsar_handler.py`

In [2]:
"""Session 1 - Handle data subject requests (access, rectification, erasure, portability).

A small SQLite database stands in for a company's customer system. The handler:
  * verifies the requester's identity before doing anything,
  * calculates the legal deadline (one month, extendable by two more for complex cases),
  * answers ACCESS and PORTABILITY requests with a machine-readable JSON export,
  * applies RECTIFICATION,
  * applies ERASURE but keeps records that a law obliges us to retain (invoices),
  * writes every action to an audit log (accountability).
All people and data are fictional. Everything runs in memory.
Run:  python session-01_dsar_handler.py
"""
import calendar
import json
import sqlite3
from datetime import date


def add_months(d, months):
    """Same day N months later; clamp to the month's last day (31 Jan + 1 month = 28/29 Feb)."""
    m = d.month - 1 + months
    y, m = d.year + m // 12, m % 12 + 1
    return date(y, m, min(d.day, calendar.monthrange(y, m)[1]))


def setup():
    db = sqlite3.connect(":memory:")
    db.executescript("""
    CREATE TABLE customers(id INTEGER PRIMARY KEY, name TEXT, email TEXT, city TEXT, phone TEXT);
    CREATE TABLE orders(id INTEGER PRIMARY KEY, customer_id INT, item TEXT, amount REAL, odate TEXT);
    CREATE TABLE invoices(id INTEGER PRIMARY KEY, customer_id INT, order_id INT, total REAL);
    CREATE TABLE marketing(customer_id INT, consent INT, segment TEXT);
    CREATE TABLE audit_log(at TEXT, request TEXT, customer_id INT, action TEXT);
    INSERT INTO customers VALUES (1,'Sara Ahmed','sara@example.com','Karachi','0300-0000001'),
                                 (2,'Bilal Khan','bilal@example.com','Lahore','0300-0000002');
    INSERT INTO orders VALUES (10,1,'Networking Basics',2500,'2026-03-02'),
                              (11,1,'Python for Security',3200,'2026-07-15'),
                              (12,2,'Digital Forensics 101',4100,'2026-05-20');
    INSERT INTO invoices VALUES (100,1,10,2500),(101,1,11,3200),(102,2,12,4100);
    INSERT INTO marketing VALUES (1,1,'tech-books'),(2,0,'none');
    """)
    return db


def log(db, today, req, cid, action):
    db.execute("INSERT INTO audit_log VALUES (?,?,?,?)", (today.isoformat(), req, cid, action))


def verify_identity(db, email, otp_ok):
    row = db.execute("SELECT id FROM customers WHERE email=?", (email,)).fetchone()
    return row[0] if row and otp_ok else None


def export(db, cid):
    """Everything we hold about one person, in a structured, commonly used format (JSON)."""
    cur = db.execute("SELECT name,email,city,phone FROM customers WHERE id=?", (cid,))
    profile = dict(zip([c[0] for c in cur.description], cur.fetchone()))
    orders = [dict(zip(["item", "amount", "date"], r)) for r in
              db.execute("SELECT item,amount,odate FROM orders WHERE customer_id=?", (cid,))]
    mk = db.execute("SELECT consent,segment FROM marketing WHERE customer_id=?", (cid,)).fetchone()
    return {"profile": profile, "orders": orders,
            "marketing": {"consent": bool(mk[0]), "segment": mk[1]} if mk else None,
            "purposes": ["order fulfilment (contract)", "newsletter (consent)"],
            "retention": "orders and invoices 6 years (tax law); marketing until consent withdrawn"}


def handle(db, req, email, otp_ok, received, complex_case=False, **kw):
    cid = verify_identity(db, email, otp_ok)
    due = add_months(received, 3 if complex_case else 1)
    print(f"\n>> {req.upper()} request from {email} received {received}  (respond by {due})")
    if cid is None:
        print("   Identity NOT verified -> ask for more proof; the clock waits. No data disclosed.")
        return
    if req in ("access", "portability"):
        data = export(db, cid)
        print(json.dumps(data, indent=2) if req == "access" else
              "   Portability file (JSON, " + str(len(json.dumps(data))) + " bytes) ready for the user to download")
    elif req == "rectification":
        db.execute("UPDATE customers SET city=? WHERE id=?", (kw["city"], cid))
        print(f"   City corrected to {kw['city']}")
    elif req == "erasure":
        db.execute("DELETE FROM marketing WHERE customer_id=?", (cid,))
        db.execute("UPDATE customers SET name='[erased]', email=NULL, phone=NULL, city=NULL WHERE id=?", (cid,))
        db.execute("UPDATE orders SET item='[erased]' WHERE customer_id=?", (cid,))
        kept = db.execute("SELECT COUNT(*) FROM invoices WHERE customer_id=?", (cid,)).fetchone()[0]
        print(f"   Profile, marketing data and order details erased.")
        print(f"   {kept} invoice(s) RETAINED: legal obligation (tax records) overrides erasure.")
    log(db, received, req, cid, "completed")


def main():
    db = setup()
    handle(db, "access", "sara@example.com", True, date(2026, 10, 1))
    handle(db, "rectification", "sara@example.com", True, date(2026, 10, 2), city="Hyderabad")
    handle(db, "portability", "bilal@example.com", True, date(2026, 1, 31))
    handle(db, "erasure", "bilal@example.com", False, date(2026, 10, 5))
    handle(db, "erasure", "bilal@example.com", True, date(2026, 10, 6), complex_case=True)
    print("\nCustomers table now:")
    for row in db.execute("SELECT * FROM customers"):
        print("  ", row)
    print("Audit log:")
    for row in db.execute("SELECT * FROM audit_log"):
        print("  ", row)


if __name__ == "__main__":
    main()


>> ACCESS request from sara@example.com received 2026-10-01  (respond by 2026-11-01)
{
  "profile": {
    "name": "Sara Ahmed",
    "email": "sara@example.com",
    "city": "Karachi",
    "phone": "0300-0000001"
  },
  "orders": [
    {
      "item": "Networking Basics",
      "amount": 2500.0,
      "date": "2026-03-02"
    },
    {
      "item": "Python for Security",
      "amount": 3200.0,
      "date": "2026-07-15"
    }
  ],
  "marketing": {
    "consent": true,
    "segment": "tech-books"
  },
  "purposes": [
    "order fulfilment (contract)",
    "newsletter (consent)"
  ],
  "retention": "orders and invoices 6 years (tax law); marketing until consent withdrawn"
}

>> RECTIFICATION request from sara@example.com received 2026-10-02  (respond by 2026-11-02)
   City corrected to Hyderabad

>> PORTABILITY request from bilal@example.com received 2026-01-31  (respond by 2026-02-28)
   Portability file (JSON, 405 bytes) ready for the user to download

>> ERASURE request from bilal@ex

## `session-01_breach_and_fines.py`

In [3]:
"""Session 1 - The 72-hour breach clock and the two tiers of GDPR fines.

Part A: a controller must notify the supervisory authority "without undue delay and,
where feasible, not later than 72 hours after having become aware" of a personal-data
breach (Article 33), unless the breach is unlikely to result in a risk to people.
High-risk breaches must ALSO be communicated to the affected people (Article 34).
Times are handled with real time zones, because the lead authority may be in a
different country from the team that finds the breach.

Part B: fines (Article 83) have two ceilings, whichever is HIGHER:
  lower tier: EUR 10 million or 2% of worldwide annual turnover of the previous year
  upper tier: EUR 20 million or 4% of worldwide annual turnover of the previous year
The ceiling is a maximum, not a price list: authorities weigh many factors.
Run:  python session-01_breach_and_fines.py
"""
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

KHI, DUB = ZoneInfo("Asia/Karachi"), ZoneInfo("Europe/Dublin")


def breach_plan(name, aware_local, records, risk):
    deadline = aware_local + timedelta(hours=72)
    print(f"\n{name}")
    print(f"  Became aware : {aware_local:%a %d %b %Y %H:%M} Karachi")
    if risk == "none":
        print("  Risk to people unlikely -> record it in the internal breach register; no notification")
        return
    print(f"  Notify DPA by: {deadline:%a %d %b %Y %H:%M} Karachi  = "
          f"{deadline.astimezone(DUB):%a %d %b %H:%M} Dublin time")
    if risk == "high":
        print(f"  High risk to {records:,} people -> ALSO tell the affected people without undue delay")


def fine_ceiling(turnover_eur, tier):
    fixed, pct = (10_000_000, 0.02) if tier == "lower" else (20_000_000, 0.04)
    return max(fixed, pct * turnover_eur)


def main():
    print("PART A - Breach notification clock")
    breach_plan("Case 1: laptop with encrypted disk stolen (key not stolen)",
                datetime(2026, 10, 23, 16, 30, tzinfo=KHI), 4_000, "none")
    breach_plan("Case 2: misconfigured cloud bucket exposed order history",
                datetime(2026, 10, 23, 16, 30, tzinfo=KHI), 12_500, "medium")
    breach_plan("Case 3: ransomware encrypted and leaked passport scans",
                datetime(2026, 10, 30, 22, 0, tzinfo=KHI), 48_000, "high")

    print("\nPART B - Maximum fine (whichever is higher)")
    print(f"  {'Company':<26}{'Turnover (EUR)':>18}{'Lower tier max':>18}{'Upper tier max':>18}")
    for name, t in [("Small online shop", 2_000_000), ("Regional bank", 900_000_000),
                    ("Global platform", 120_000_000_000)]:
        lo, hi = fine_ceiling(t, "lower"), fine_ceiling(t, "upper")
        print(f"  {name:<26}{t:>18,.0f}{lo:>18,.0f}{hi:>18,.0f}")
    print("\n  Break-even: 4% beats the EUR 20m floor when turnover > EUR "
          f"{20_000_000 / 0.04:,.0f}")


if __name__ == "__main__":
    main()

PART A - Breach notification clock

Case 1: laptop with encrypted disk stolen (key not stolen)
  Became aware : Fri 23 Oct 2026 16:30 Karachi
  Risk to people unlikely -> record it in the internal breach register; no notification

Case 2: misconfigured cloud bucket exposed order history
  Became aware : Fri 23 Oct 2026 16:30 Karachi
  Notify DPA by: Mon 26 Oct 2026 16:30 Karachi  = Mon 26 Oct 11:30 Dublin time

Case 3: ransomware encrypted and leaked passport scans
  Became aware : Fri 30 Oct 2026 22:00 Karachi
  Notify DPA by: Mon 02 Nov 2026 22:00 Karachi  = Mon 02 Nov 17:00 Dublin time
  High risk to 48,000 people -> ALSO tell the affected people without undue delay

PART B - Maximum fine (whichever is higher)
  Company                       Turnover (EUR)    Lower tier max    Upper tier max
  Small online shop                  2,000,000        10,000,000        20,000,000
  Regional bank                    900,000,000        18,000,000        36,000,000
  Global platform           

## `session-01_pseudonymise.py`

In [4]:
"""Session 1 - Masking, hashing, keyed pseudonymisation and k-anonymity.

GDPR encourages "pseudonymisation": replacing identifiers so data can no longer be
linked to a person WITHOUT extra information that is kept separately and securely.
Pseudonymised data is STILL personal data. Truly anonymous data is not.

This script compares four techniques on fictional student records and shows why a
plain unsalted hash of a phone number is weak (it can be reversed by trying every
possible number), while a keyed HMAC with a secret key is much stronger.
Run:  python session-01_pseudonymise.py
"""
import hashlib
import hmac
import secrets
from collections import Counter

RECORDS = [  # fictional
    {"name": "Ayesha Raza", "phone": "03001234567", "age": 21, "city": "Karachi", "grade": "A"},
    {"name": "Hamza Ali",   "phone": "03001234568", "age": 22, "city": "Karachi", "grade": "B"},
    {"name": "Zara Shah",   "phone": "03111234569", "age": 21, "city": "Karachi", "grade": "A"},
    {"name": "Omar Siddiqui", "phone": "03211234570", "age": 34, "city": "Quetta", "grade": "C"},
    {"name": "Fatima Noor", "phone": "03001234571", "age": 23, "city": "Karachi", "grade": "B"},
]


def mask(phone):
    return phone[:4] + "*****" + phone[-2:]


def plain_hash(phone):
    return hashlib.sha256(phone.encode()).hexdigest()[:16]


def keyed(phone, key):
    return hmac.new(key, phone.encode(), hashlib.sha256).hexdigest()[:16]


def brute_force_plain(target, prefix="0300123", digits=4):
    """Try every number with a known prefix - shows why unsalted hashes of phones are weak."""
    for n in range(10 ** digits):
        guess = f"{prefix}{n:0{digits}d}"
        if plain_hash(guess) == target:
            return guess, n + 1
    return None, 10 ** digits


def generalise(r):
    """Coarsen quasi-identifiers: exact age -> age band, keep city."""
    lo = r["age"] // 10 * 10
    return (f"{lo}-{lo + 9}", r["city"])


def main():
    # TEACHING ONLY: a fixed key so the output is repeatable. In real life generate it with
    # secrets.token_bytes(32) and keep it in a key vault, away from the pseudonymised data.
    key = b"demo-key-0123456789abcdef-0123456"
    print(f"{'Name':<15}{'Masked':<14}{'SHA-256 (plain)':<18}{'HMAC (keyed)':<18}")
    for r in RECORDS:
        print(f"{r['name']:<15}{mask(r['phone']):<14}{plain_hash(r['phone']):<18}{keyed(r['phone'], key):<18}")
    same = keyed(RECORDS[0]["phone"], key) == keyed(RECORDS[0]["phone"], key)
    print(f"\nHMAC is consistent with the same key (records can still be joined): {same}")
    print("HMAC with a different key gives a different pseudonym:",
          keyed(RECORDS[0]["phone"], key) != keyed(RECORDS[0]["phone"], secrets.token_bytes(32)))

    target = plain_hash(RECORDS[4]["phone"])
    found, tries = brute_force_plain(target)
    print(f"\nAttack on the plain hash {target}: recovered {found} after {tries:,} guesses")
    print("Same attack on the HMAC is not possible without the secret key.")

    print("\nk-anonymity after generalising (age band, city):")
    groups = Counter(generalise(r) for r in RECORDS)
    for g, k in sorted(groups.items()):
        flag = "  <- unique: person can be singled out" if k == 1 else ""
        print(f"   {g[0]:<7} {g[1]:<8} k={k}{flag}")
    print(f"Dataset k = {min(groups.values())}  (want k >= 2, ideally much higher, before sharing)")


if __name__ == "__main__":
    main()

Name           Masked        SHA-256 (plain)   HMAC (keyed)      
Ayesha Raza    0300*****67   f81b06138a382c85  f199178130750c86  
Hamza Ali      0300*****68   eda002d67093674c  9f6e1aa2e59ca2f1  
Zara Shah      0311*****69   6f025643f7aa77d5  7dcb2b1eacbd6cf4  
Omar Siddiqui  0321*****70   444b40c4492f461e  13e9a5c1a3760a31  
Fatima Noor    0300*****71   97177fe8bfc08294  c47e3c0f7ea947ec  

HMAC is consistent with the same key (records can still be joined): True
HMAC with a different key gives a different pseudonym: True

Attack on the plain hash 97177fe8bfc08294: recovered 03001234571 after 4,572 guesses
Same attack on the HMAC is not possible without the secret key.

k-anonymity after generalising (age band, city):
   20-29   Karachi  k=4
   30-39   Quetta   k=1  <- unique: person can be singled out
Dataset k = 1  (want k >= 2, ideally much higher, before sharing)
